# Purchase Propensity Modeling

## Business Question

Can we identify sessions that are more likely to result in a purchase
using information available at the start of the session?

## Prediction Setup

- Prediction unit: Session
- Target: `purchased`
- Prediction moment: Session start
- Positive class: 1 = Purchase
- Negative class: 0 = No purchase



## Business Use

The goal is to identify high-propensity sessions for targeted,
relevant interventions rather than treating every session equally.

In [23]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    average_precision_score,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("BigQuerySQLAnalysis.csv")
df.head(3)

,user_pseudo_id,ga_session_id,session_number,session_date,session_start,session_end,session_duration_seconds,event_count,page_views,product_views,...,purchased,purchase_revenue,device_category,operating_system,browser,country,city,acquisition_source,acquisition_medium,user_type
0,1.032845e+06,3019986733,1,2020-12-25,2020-12-25 22:14:45.725795 UTC,2020-12-25 22:17:28.019108 UTC,162,19,7,1,...,0,0.0,desktop,Macintosh,Chrome,United States,(not set),google,organic,New
1,1.033347e+06,734524201,1,2020-12-25,2020-12-25 03:58:00.180345 UTC,2020-12-25 03:58:00.180345 UTC,0,3,1,0,...,0,0.0,mobile,Web,Safari,Jordan,(not set),google,organic,New
2,1.058591e+06,9177820994,1,2020-12-25,2020-12-25 05:11:50.59229 UTC,2020-12-25 05:12:05.924445 UTC,15,4,1,0,...,0,0.0,desktop,Web,Chrome,United States,Minneapolis,google,organic,New


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 360129 entries, 0 to 360128
Data columns (total 26 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   user_pseudo_id            360129 non-null  float64
 1   ga_session_id             360129 non-null  int64  
 2   session_number            360129 non-null  int64  
 3   session_date              360129 non-null  object 
 4   session_start             360129 non-null  object 
 5   session_end               360129 non-null  object 
 6   session_duration_seconds  360129 non-null  int64  
 7   event_count               360129 non-null  int64  
 8   page_views                360129 non-null  int64  
 9   product_views             360129 non-null  int64  
 10  cart_adds                 360129 non-null  int64  
 11  viewed_product            360129 non-null  int64  
 12  added_to_cart             360129 non-null  int64  
 13  started_checkout          360129 non-null  i

In [6]:
df["session_date"] = pd.to_datetime(df["session_date"])

print("Shape:", df.shape)
print("Purchase rate:", df["purchased"].mean())
print("Date range:", df["session_date"].min().date(),
     "to", df["session_date"].max().date())

Shape: (360129, 26)
Purchase rate: 0.013461842839649126
Date range: 2020-11-01 to 2021-01-31


#### Model is dealing with Imbalanced target
- Only ~1.35% of sessions purchased

In [14]:
data = df.copy()

data["day_of_week"] = data["session_date"].dt.dayofweek
data["month"] = data["session_date"].dt.month
data["is_weekend"] = (
    data["day_of_week"] >= 5
).astype(int)

features = [
    "session_number",
    "day_of_week",
    "month",
    "is_weekend",
    "device_category",
    "operating_system",
    "browser",
    "country",
    "acquisition_source",
    "acquisition_medium"
]

target = "purchased"

model_df = data[features + [target, "session_date"]].copy()

print("Model dataset shape:", model_df.shape)
model_df.head()

Model dataset shape: (360129, 12)


,session_number,day_of_week,month,is_weekend,device_category,operating_system,browser,country,acquisition_source,acquisition_medium,purchased,session_date
0,1,4,12,0,desktop,Macintosh,Chrome,United States,google,organic,0,2020-12-25
1,1,4,12,0,mobile,Web,Safari,Jordan,google,organic,0,2020-12-25
2,1,4,12,0,desktop,Web,Chrome,United States,google,organic,0,2020-12-25
3,1,4,12,0,desktop,Web,Edge,China,(direct),(none),0,2020-12-25
4,2,4,12,0,desktop,Web,Chrome,United States,(data deleted),(data deleted),0,2020-12-25


In [15]:
cutoff_date = model_df["session_date"].max() - pd.Timedelta(days=20)

train_df = model_df[model_df["session_date"] < cutoff_date].copy()
test_df = model_df[model_df["session_date"] >= cutoff_date].copy()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain purchase rate:", y_train.mean())
print("Test purchase rate:", y_test.mean())

print("\nTest period:",
      test_df["session_date"].min().date(),
      "to",
      test_df["session_date"].max().date())

Train shape: (278602, 10)
Test shape: (81527, 10)

Train purchase rate: 0.014127680346874753
Test purchase rate: 0.011186478099279994

Test period: 2021-01-11 to 2021-01-31


In [17]:
numeric_features = [
    "session_number",
    "day_of_week",
    "month",
    "is_weekend"
]

categorical_features = [
    "device_category",
    "operating_system",
    "browser",
    "country",
    "acquisition_source",
    "acquisition_medium"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            "passthrough",
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)

logistic_probs = logistic_model.predict_proba(X_test)[:, 1]
logistic_preds = (logistic_probs >= 0.05).astype(int)

In [24]:
roc_auc = roc_auc_score(y_test, logistic_probs)
pr_auc = average_precision_score(y_test, logistic_probs)
precision = precision_score(y_test, logistic_preds, zero_division=0)
recall = recall_score(y_test, logistic_preds, zero_division=0)

print(f"ROC-AUC: {roc_auc:.3f}")
print(f"PR-AUC: {pr_auc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")

ROC-AUC: 0.689
PR-AUC: 0.029
Precision: 0.011
Recall: 0.993


In [27]:
from xgboost import XGBClassifier

In [29]:
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=42
                )
        )
    ]
)

xgb_model.fit(X_train, y_train)

xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

In [30]:
xgb_roc_auc = roc_auc_score(y_test, xgb_probs)
xgb_pr_auc = average_precision_score(y_test, xgb_probs)


print(f"XGBoost ROC-AUC: {xgb_roc_auc:.3f}")
print(f"XGBoost PR-AUC: {xgb_pr_auc:.3f}")

XGBoost ROC-AUC: 0.716
XGBoost PR-AUC: 0.031


In [31]:
eval_df = X_test.copy()

eval_df["actual"] = y_test.values
eval_df["score"] = xgb_probs

eval_df = eval_df.sort_values("score", ascending=False)

# Top 10%

n_top = int(len(eval_df) * 0.10)
top10 = eval_df.head(n_top)

overall_rate = eval_df["actual"].mean()
top10_rate = top10["actual"].mean()

lift = top10_rate / overall_rate

captured_purchase_share = (
    top10["actual"].sum() / eval_df["actual"].sum()
)

print(f"Test sessions: {len(eval_df):,}")
print(f"Overall purchase rate: {overall_rate:.2%}")
print(f"Top 10% purchase rate: {top10_rate:.2%}")
print(f"Top 10% lift: {lift:.2f}x")
print(f"Purchases captured by top 10%: {captured_purchase_share:.2%}")

Test sessions: 81,527
Overall purchase rate: 1.12%
Top 10% purchase rate: 4.02%
Top 10% lift: 3.60x
Purchases captured by top 10%: 35.96%


### XGBoost Evaluation

XGBoost achieved:

- **ROC-AUC:** 0.716
- **PR-AUC:** 0.031

At first glance, the PR-AUC of 0.031 may appear low. However, purchase is a highly imbalanced outcome: only **1.12% of sessions in the test set resulted in a purchase**.

Therefore, the random baseline for PR-AUC is approximately **0.011** (the positive-class prevalence). The model's PR-AUC of **0.031 is substantially above this baseline**, indicating that the model has useful ability to concentrate likely purchasers near the top of its ranking.

The business impact is clearer from the lift analysis:

- Overall purchase rate: **1.12%**
- Top 10% propensity sessions: **4.02%**
- **Top-decile lift: 3.60×**
- **35.96% of all purchases captured in the top 10% of sessions**

This means the model is not accurate enough to be treated as a perfect purchase predictor, but it is useful as a **ranking model**. By focusing an intervention on the highest-propensity 10% of sessions, the business could concentrate attention on a segment whose observed purchase rate is approximately **3.6× the overall test-set rate**.

Therefore, the model's primary value is **prioritization rather than exact prediction**.

In [33]:
# XGBoost Feature Importance

feature_names = xgb_model.named_steps["preprocessor"].get_feature_names_out()
importances = xgb_model.named_steps["model"].feature_importances_

feature_importance = (
    pd.Series(importances, index=feature_names)
    .sort_values(ascending=False)
    .head(15)
)
feature_importance

num__session_number                                        0.127143
cat__acquisition_source_(data deleted)                     0.038384
cat__acquisition_medium_(data deleted)                     0.029450
num__month                                                 0.029033
cat__acquisition_source_shop.googlemerchandisestore.com    0.023027
cat__country_Colombia                                      0.013297
cat__country_United Kingdom                                0.012570
cat__country_Slovenia                                      0.012185
cat__operating_system_Macintosh                            0.011582
cat__country_Vietnam                                       0.011557
cat__country_United States                                 0.011536
cat__country_Philippines                                   0.011036
cat__browser_Edge                                          0.010651
cat__country_Malaysia                                      0.010175
cat__country_Spain                              

### Feature Importance

The XGBoost model's feature importance suggests that **session history, acquisition characteristics, and user/session context** contributed to purchase propensity.

The strongest individual feature was `session_number`, followed by acquisition source/medium variables and month. Several geographic, operating-system, and browser categories also appeared among the top individual encoded features.

However, these importances should be interpreted as **predictive signals rather than causal drivers**. In particular, categorical variables are one-hot encoded, so importance is distributed across individual categories.

The model therefore helps identify sessions with different levels of purchase propensity, but it does not establish why those sessions convert differently.

The main takeaway is that **early-session context contains enough signal to rank sessions by purchase propensity**, which supports using the model for prioritization and targeted experimentation rather than deterministic purchase prediction.

In [34]:
print("MODEL SUMMARY")
print("-" * 40)
print(f"Logistic Regression ROC-AUC: {roc_auc:.3f}")
print(f"Logistic Regression PR-AUC:  {pr_auc:.3f}")
print(f"XGBoost ROC-AUC:            {xgb_roc_auc:.3f}")
print(f"XGBoost PR-AUC:             {xgb_pr_auc:.3f}")
print(f"Top 10% Lift:               {lift:.2f}x")
print(f"Top 10% Purchase Capture:   {captured_purchase_share:.2%}")

MODEL SUMMARY
----------------------------------------
Logistic Regression ROC-AUC: 0.689
Logistic Regression PR-AUC:  0.029
XGBoost ROC-AUC:            0.716
XGBoost PR-AUC:             0.031
Top 10% Lift:               3.60x
Top 10% Purchase Capture:   35.96%
